In [107]:
import sys ; sys.argv = ['']
import template
from option import parser
import utility
import model
import imageio

def get_args(parser, arglist):
    args = parser.parse_args(arglist)
    template.set_template(args)

    args.scale = list(map(lambda x: int(x), args.scale.split('+')))
    args.input_range = [float(x.strip('\"\'')) for x in args.input_range.split(',')]
    args.tensor_range = [float(x.strip('\"\'')) for x in args.tensor_range.split(',')]
    args.data_train = args.data_train.split('+')
    args.data_test = args.data_test.split('+')

    if args.epochs == 0:
        args.epochs = 1e8

    for arg in vars(args):
        if vars(args)[arg] == 'True':
            vars(args)[arg] = True
        elif vars(args)[arg] == 'False':
            vars(args)[arg] = False
            
    return args

In [132]:
argdict = {
    'SCALE' : 2,
    'TEMPLATE' : 'EDSR_paper',
    'DATASET' : 'DIV2K_TIFF'
}

argstring = \
"""
--test_only
--template {TEMPLATE}
--scale {SCALE}
--shift_mean False
--dir_data ../../Data 
--data_train {DATASET}
--data_test {DATASET}
--data_range 1-800/1-10
--input_range '-1.,1.'
--tensor_range '-1.,1.'
--n_colors 1
--no_augment
--ext img""".format(**argdict)

arglist = [i for a in argstring.split('\n') for i in a.split()]
sys.argv = arglist

args = get_args(parser, arglist)

In [123]:
from data import div2k_tiff

In [146]:
import data

data_loader = data.Data(args)

In [141]:

checkpoint = utility.checkpoint(args)
_model = model.Model(args, checkpoint)


Making model EDSR...


In [135]:
img = data_loader[0]

In [139]:
lr = img[0]

In [140]:
lr.shape

torch.Size([1, 678, 1020])

In [120]:
import numpy as np

imgfile = '../../Data/DIV2K_TIFF/DIV2K_train_LR_bicubic/X2/0001x2.tiff'
device = torch.device('cuda')

img = imageio.imread(imgfile)
img = np.expand_dims(img, axis=2)
lr = torch.tensor(img, device=device)

In [145]:
_model.eval()
sr = _model(lr,0)

RuntimeError: Expected 4-dimensional input for 4-dimensional weight 256 1 3 3, but got 3-dimensional input of size [1, 678, 1020] instead

In [144]:
help(_model.eval)

Help on method eval in module torch.nn.modules.module:

eval() method of model.Model instance
    Sets the module in evaluation mode.
    
    This has any effect only on certain modules. See documentations of
    particular modules for details of their behaviors in training/evaluation
    mode, if they are affected, e.g. :class:`Dropout`, :class:`BatchNorm`,
    etc.
    
    This is equivalent with :meth:`self.train(False) <torch.nn.Module.train>`.
    
    Returns:
        Module: self



In [161]:
d = data_loader.loader_test[0]

In [162]:
d.dataset.set_scale(2)

In [164]:
for img in d:
    print(img)

IndexError: Caught IndexError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/opt/conda/lib/python3.8/site-packages/torch/utils/data/_utils/worker.py", line 178, in _worker_loop
    data = fetcher.fetch(index)
  File "/opt/conda/lib/python3.8/site-packages/torch/utils/data/_utils/fetch.py", line 44, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/opt/conda/lib/python3.8/site-packages/torch/utils/data/_utils/fetch.py", line 44, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/home/msc2018-ceb/ra024374/Experimentos/edsr-pytorch/src/data/srdata.py", line 99, in __getitem__
    lr, hr, filename = self._load_file(idx)
  File "/home/msc2018-ceb/ra024374/Experimentos/edsr-pytorch/src/data/div2k_tiff.py", line 53, in _load_file
    f_lr = self.images_lr[self.idx_scale][idx]
IndexError: list index out of range
